# Part 2

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers accelerate huggingface_hub pillow pypdfium2 codecarbon

In [ ]:
import os
import json
import time
from datetime import datetime
from pathlib import Path

import torch
import pandas as pd
from PIL import Image
import pypdfium2 as pdfium
from transformers import AutoProcessor, AutoModelForImageTextToText

from codecarbon import EmissionsTracker


from huggingface_hub import login
login("hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# =========================
# CONFIG
# =========================
MODEL_ID = "google/gemma-3-4b-it"   # change this later for other VLMs

NUM_RUNS = 3
CPU_WARMUP_SECS = 300   # 5 min
GPU_WARMUP_SECS = 60    # 1 min
COOLDOWN_SECS = 60      # 1 min
MEASURE_POWER_SECS = 1

MAIN_OUTPUT_DIR = "/content/drive/MyDrive/UML_CODECARBON_PhaseWise"

# Input should come from Part 1 output folder
PDF_DIR = "/content/drive/MyDrive/UML_CODECARBON_PhaseWise/phase1_part1_fixed/uml_pages_pdfs"

MODEL_TAG = MODEL_ID.replace("/", "__")
PART2_ROOT = os.path.join(MAIN_OUTPUT_DIR, "phase1_part2_runs", MODEL_TAG)

os.makedirs(PART2_ROOT, exist_ok=True)

def backup_if_exists(path):
    if os.path.exists(path):
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        new_name = f"{path}.old_{ts}"
        os.rename(path, new_name)
        print(f"Existing file backed up: {new_name}")

def save_json(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def make_tracker(output_dir: str, project_name: str, output_file: str):
    os.makedirs(output_dir, exist_ok=True)
    backup_if_exists(os.path.join(output_dir, output_file))
    return EmissionsTracker(
        project_name=project_name,
        output_dir=output_dir,
        output_file=output_file,
        measure_power_secs=MEASURE_POWER_SECS,
        log_level="warning",
    )

In [ ]:
setup_tracking_dir = os.path.join(PART2_ROOT, "setup_tracking")

setup_tracker = make_tracker(
    output_dir=setup_tracking_dir,
    project_name="phase1_part2_setup",
    output_file="phase1_part2_setup_emissions.csv"
)

print("PART 2 SETUP TRACKER START")
setup_tracker.start()
setup_t0 = time.time()

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

setup_emissions = setup_tracker.stop()
setup_duration = time.time() - setup_t0
print("PART 2 SETUP TRACKER STOP")

setup_summary = {
    "phase": "phase1_part2",
    "model_id": MODEL_ID,
    "setup_duration_sec": setup_duration,
    "setup_emissions_kg": setup_emissions,
    "device": device,
    "input_pdf_dir": PDF_DIR
}
save_json(setup_summary, os.path.join(setup_tracking_dir, "phase1_part2_setup_summary.json"))

print("Loaded model on:", device)
setup_summary

In [ ]:
def warmup_cpu(duration_secs):
    print(f"\nCPU warm-up for {duration_secs} seconds...")
    start = time.time()
    a, b = 0, 1
    while (time.time() - start) < duration_secs:
        a, b = b, a + b
        if a > 10**7:
            a, b = 0, 1
    print("CPU warm-up finished.\n")

@torch.no_grad()
def warmup_part2_vlm_once():
    dummy_img = Image.new("RGB", (256, 256), color="white")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": "Describe this image in one short line."},
            ],
        }
    ]

    text_in = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = processor(
        text=[text_in],
        images=[dummy_img],
        return_tensors="pt",
    ).to(device, torch.float16)

    _ = model.generate(
        **inputs,
        max_new_tokens=16,
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

def run_part2_warmup():
    warmup_cpu(CPU_WARMUP_SECS)

    print(f"GPU/VLM warm-up for {GPU_WARMUP_SECS} seconds...")
    start = time.time()
    while (time.time() - start) < GPU_WARMUP_SECS:
        warmup_part2_vlm_once()
        time.sleep(2)
    print("GPU/VLM warm-up finished.\n")

print("\n========== PART 2 WARM-UP START (EXCLUDED) ==========")
run_part2_warmup()
print("========== PART 2 WARM-UP STOP ==========\n")

# Prompt

In [ ]:
def pdf_to_page_images(pdf_path, out_dir, scale=2.0):
    """
    Render each page of the PDF to a PNG image.
    Returns list of image paths in order.
    """
    os.makedirs(out_dir, exist_ok=True)
    import pypdfium2 as pdfium

    pdf = pdfium.PdfDocument(pdf_path)
    img_paths = []
    for i in range(len(pdf)):
        page = pdf[i]
        bitmap = page.render(scale=scale)
        img = bitmap.to_pil()
        out_path = os.path.join(
            out_dir,
            f"{Path(pdf_path).stem}_page_{i+1:03d}.png"
        )
        img.save(out_path)
        img_paths.append(out_path)
    return img_paths


UML_TYPES = [
    "class",
    "activity",
    "sequence",
    "communication",
    "component",
    "deployment",
    "object",
    "package",
    "state_machine",
    "use_case",
    "other",
]


def describe_uml_image(image_path: str):
    """
    Generate a rich *text* description for ONE UML diagram image.
    Output is structured plain text (no JSON) so you can later
    use it for test-case generation.
    Now includes a final 'Observations' section as a global summary.
    Returns:
        result, input_token_count, output_token_count
    """
    from PIL import Image

    image = Image.open(image_path).convert("RGB")

    prompt = f"""
You are a precise UML diagram analysis assistant.

You will see ONE UML diagram image. It may be any of these UML types:
{", ".join(UML_TYPES)}.

Your task is to READ ONLY what is visible in the diagram and produce a
DETAILED BUT STRUCTURED description that will later be used to generate
test cases and test scripts.

The output must strictly follow this structure (in this order, no extra
headings before or after, no system/user/assistant tags):

UML Type:
- <one of: {", ".join(UML_TYPES)}>

Components:
- <Name> — kind=<kind>; attributes=[a: Type, ...]; operations=[op(args): Ret, ...]; role=<short role/meaning using only visible labels>

Actors / Lifelines:
- <actor or lifeline name 1>
- <actor or lifeline name 2>
  (if none are explicitly shown, write "- None")

Relationships / Links:
- <Source> -> <Target> — type=<association/message/transition/etc>; label="<edge label if visible>"; explanation="<short explanation using only visible text>"

Main Flows (step by step):
- Flow 1: <step-by-step description from start to end using only component names and edge labels>
- Flow 2: <only if another distinct flow is clearly visible>
  (you can add Flow 3, Flow 4 if they clearly exist; otherwise stop at Flow 1)

Conditions / Branches:
- <describe each decision/guard and its outgoing labeled branches, using ONLY labels visible in the diagram>
  (if there are no decisions, write "- None")

Loops / Repetitions:
- <describe any loops or repeated behaviour where arrows return to earlier nodes or messages repeat>
  (if none, write "- None")

Notes / Annotations:
- <any textual notes, stereotypes, constraints, or comments shown>
  (if none, write "- None")

Observations:
- <2–6 bullets giving an overall high-level description of what this diagram models,
  how the main flows behave, and any key edge cases or error paths, using ONLY
  information from labels in the diagram. Think of this as a concise summary a
  tester will read before designing test cases.>

Guidelines by diagram type (KEEP IT GENERIC):

- For CLASS diagrams:
  * Treat each class box as a component with kind="class".
  * In the component bullet, list all attributes and operations exactly as written,
    including type and return type if visible.
  * In Relationships / Links, list associations, aggregations, compositions,
    and inheritances between classes (use "association", "inheritance", etc.).
  * Main Flows can describe likely collaboration between classes based only
    on visible methods and relationships (no guessing new names).

- For ACTIVITY diagrams:
  * Components are actions, decisions, merges, initial nodes, final nodes, etc.
  * Relationships / Links correspond to control-flow arrows.
  * Main Flows should be explicit sequences like:
    "Avvio app" -> "Utente registrato con successo al servizio?" (SI) -> ...
  * Conditions / Branches summarize each decision node and its labeled exits.

- For SEQUENCE diagrams:
  * Actors / Lifelines should list all participants (Client, Server, Database, Logger, etc.).
  * Components can repeat these lifelines plus important fragments (alt, loop, etc.).
  * Relationships / Links must capture each message with its order and arrow direction.
  * Main Flows should narrate the interaction from top to bottom.

- For USE-CASE diagrams:
  * Actors are external users or systems.
  * Components include use cases (ellipses) and actors.
  * Relationships / Links capture "includes", "extends", associations, etc.

- For STATE_MACHINE diagrams:
  * Components are states and initial/final markers.
  * Relationships / Links are transitions with event/guard labels.

- For COMPONENT / DEPLOYMENT / OBJECT / PACKAGE diagrams:
  * Components are components, nodes, objects, or packages respectively.
  * Relationships / Links describe connections, dependencies, deployments, or containment.

GENERAL RULES (VERY IMPORTANT):
- Use ONLY text that is visible in the diagram. Do NOT invent new names.
- If something is not present, write "- None" or use an empty [] in that bullet.
- For components, always try to include: name, kind, attributes (if visible),
  operations (if visible), and a short role/meaning in plain language.
- Be as detailed and explicit as possible while staying faithful to the diagram.
- Do NOT repeat these instructions in the output; start directly with:

  UML Type:
  - ...
"""

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt.strip()},
            ],
        }
    ]

    text_in = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    input_token_count = None
    if hasattr(processor, "tokenizer") and processor.tokenizer is not None:
        input_token_count = len(processor.tokenizer.encode(text_in, add_special_tokens=False))

    inputs = processor(
        text=[text_in],
        images=[image],
        return_tensors="pt",
    ).to(device, torch.float16)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=900,
        )

    result = processor.batch_decode(output_ids, skip_special_tokens=True)[0].strip()

    output_token_count = None
    if hasattr(processor, "tokenizer") and processor.tokenizer is not None:
        output_token_count = len(processor.tokenizer.encode(result, add_special_tokens=False))

    lower = result.lower()
    key = "uml type:"
    if key in lower:
        cut_idx = lower.index(key)
        result = result[cut_idx:]

    return result, input_token_count, output_token_count

In [ ]:
def run_part2_single(run_id: int):
    run_root = os.path.join(PART2_ROOT, f"run_{run_id}")
    os.makedirs(run_root, exist_ok=True)

    IMG_ROOT_DIR = os.path.join(run_root, "uml_page_images_all_reports")
    OUTPUT_TXT = os.path.join(run_root, "ALL_uml_reports_descriptions.txt")
    TOKEN_CSV = os.path.join(run_root, "phase1_part2_token_counts.csv")
    TRACKING_DIR = os.path.join(run_root, "inference_tracking")

    os.makedirs(IMG_ROOT_DIR, exist_ok=True)
    os.makedirs(TRACKING_DIR, exist_ok=True)

    tracker = make_tracker(
        output_dir=TRACKING_DIR,
        project_name="phase1_part2_inference",
        output_file=f"phase1_part2_run_{run_id}_inference_emissions.csv"
    )

    print(f"\nPART 2 RUN {run_id} INFERENCE TRACKER START")
    tracker.start()
    infer_t0 = time.time()

    all_lines = []
    token_rows = []

    pdf_files = sorted([
        f for f in Path(PDF_DIR).glob("*.pdf")
    ])

    print(f"Found {len(pdf_files)} PDF reports\n")

    for pdf_idx, pdf_path in enumerate(pdf_files, start=1):
        print(f"\n==============================")
        print(f"Processing report {pdf_idx}/{len(pdf_files)}")
        print(f"REPORT: {pdf_path.name}")
        print(f"==============================")

        report_img_dir = os.path.join(
            IMG_ROOT_DIR,
            pdf_path.stem
        )

        page_images = pdf_to_page_images(
            str(pdf_path),
            report_img_dir,
            scale=2.0
        )

        print(f"Extracted {len(page_images)} page images")

        all_lines.append("\n" + "=" * 70)
        all_lines.append(f"REPORT: {pdf_path.name}")
        all_lines.append("=" * 70 + "\n")

        for page_idx, img_path in enumerate(sorted(page_images), start=1):
            print(f" -> Page {page_idx}/{len(page_images)}")

            desc, in_tok, out_tok = describe_uml_image(img_path)

            block = []
            block.append(f"===== {pdf_path.name} | Page {page_idx} =====")
            block.append(desc.strip())
            block.append("")

            all_lines.append("\n".join(block))

            token_rows.append({
                "phase": "phase1_part2",
                "run_id": run_id,
                "report": pdf_path.name,
                "page": page_idx,
                "input_tokens": in_tok,
                "output_tokens": out_tok,
                "total_tokens": (in_tok or 0) + (out_tok or 0),
                "source_image": img_path
            })

    full_text = "\n".join(all_lines)

    with open(OUTPUT_TXT, "w", encoding="utf-8") as f:
        f.write(full_text)

    token_df = pd.DataFrame(token_rows)
    token_df.to_csv(TOKEN_CSV, index=False)

    inference_emissions = tracker.stop()
    inference_duration = time.time() - infer_t0
    print(f"PART 2 RUN {run_id} INFERENCE TRACKER STOP")

    inference_summary = {
        "phase": "phase1_part2",
        "run_id": run_id,
        "mode": "repeated_runs",
        "model_id": MODEL_ID,
        "input_pdf_dir": PDF_DIR,
        "output_txt": OUTPUT_TXT,
        "token_csv": TOKEN_CSV,
        "inference_duration_sec": inference_duration,
        "inference_emissions_kg": inference_emissions,
    }

    save_json(
        inference_summary,
        os.path.join(TRACKING_DIR, f"phase1_part2_run_{run_id}_inference_summary.json")
    )

    print("\n✅ All reports processed")
    print("📄 Saved combined UML descriptions to:", OUTPUT_TXT)

    return inference_summary

In [ ]:
all_run_summaries = []

for run_id in range(1, NUM_RUNS + 1):
    print(f"\n==================== PART 2 RUN {run_id}/{NUM_RUNS} ====================")
    run_summary = run_part2_single(run_id)
    all_run_summaries.append(run_summary)

    if run_id < NUM_RUNS:
        print(f"\nCooldown for {COOLDOWN_SECS} seconds...\n")
        time.sleep(COOLDOWN_SECS)

# ***code for cleaning the report***

In [ ]:
import re
from pathlib import Path

def clean_page_block(page_block: str) -> str:
    """
    Given the raw text for ONE page block (including template),
    returns only the final real UML description.
    """

    # Normalize newlines
    s = page_block.replace("\r\n", "\n")

    # 1) Prefer the LAST occurrence of "assistant\nUML Type:" (real model output)
    marker = "\nassistant\nUML Type:"
    idx = s.rfind(marker)

    if idx != -1:
        cleaned = s[idx + len("\nassistant\n"):]   # remove the "assistant" line
        return cleaned.strip()

    # 2) Fallback: keep from the LAST "UML Type:" (if assistant marker is absent)
    idx2 = s.rfind("\nUML Type:")
    if idx2 != -1:
        cleaned = s[idx2 + 1:]  # +1 to drop leading newline
        return cleaned.strip()

    # 3) If nothing matches, return original (rare)
    return s.strip()


def clean_report_txt(in_path: str, out_path: str):
    text = Path(in_path).read_text(encoding="utf-8", errors="ignore")
    text = text.replace("\r\n", "\n")

    # Split by pages (keeps the page header tokens)
    # Example header: "===== Report_10_uml_pages.pdf | Page 1 ====="
    parts = re.split(r"(===== .*? \| Page \d+ =====\n)", text)

    # parts structure: [prefix, header1, body1, header2, body2, ...]
    if len(parts) < 3:
        raise ValueError("No page blocks found. Check the input formatting.")

    prefix = parts[0]  # includes REPORT header lines
    cleaned_chunks = [prefix.strip()]

    for i in range(1, len(parts), 2):
        header = parts[i].rstrip("\n")
        body = parts[i + 1] if i + 1 < len(parts) else ""

        cleaned_body = clean_page_block(body)

        cleaned_chunks.append(header)
        cleaned_chunks.append(cleaned_body)
        cleaned_chunks.append("")  # blank line between pages

    Path(out_path).write_text("\n".join(cleaned_chunks).strip() + "\n", encoding="utf-8")
    print("Saved cleaned file to:", out_path)


# -------------------------
# Example usage (single report file)
# -------------------------
IN_TXT  = "/content/drive/MyDrive/UML_CODECARBON_PhaseWise/phase1_part2_runs/Qwen__Qwen2.5-VL-7B-Instruct/run_3/ALL_uml_reports_descriptions.txt"   # change
OUT_TXT = "/content/drive/MyDrive/UML_CODECARBON_PhaseWise/phase1_part2_runs/Qwen__Qwen2.5-VL-7B-Instruct/run_3/AllUMLReportsDescriptions_CLEANED.txt"                            # output

clean_report_txt(IN_TXT, OUT_TXT)


# ***Separate txt files for each report***

In [ ]:
import re
from pathlib import Path

# =========================
# CONFIG
# =========================
IN_TXT = "/content/drive/MyDrive/UML_CODECARBON_PhaseWise/phase1_part2_runs/Qwen__Qwen2.5-VL-7B-Instruct/run_3/AllUMLReportsDescriptions_CLEANED.txt"   # <-- change if your big file is elsewhere
OUT_ROOT = Path("/content/drive/MyDrive/UML_CODECARBON_PhaseWise/phase1_part2_runs/Qwen__Qwen2.5-VL-7B-Instruct/run_3/REPORTS_SPLIT")               # main folder
OUT_SUB  = OUT_ROOT / "reports_txt"                     # subfolder (as you asked)
OUT_SUB.mkdir(parents=True, exist_ok=True)

text = Path(IN_TXT).read_text(encoding="utf-8", errors="ignore").replace("\r\n", "\n")

# =========================
# SPLIT LOGIC (report-wise)
# =========================
# Matches blocks that start at: "REPORT: <something>"
# and continues until the next "REPORT:" or end-of-file.
pattern = re.compile(
    r"(?ms)^\s*REPORT:\s*(?P<report>.+?)\s*\n(?P<body>.*?)(?=^\s*REPORT:\s*.+?\s*\n|\Z)"
)

matches = list(pattern.finditer(text))
print(f"Found {len(matches)} reports")

if not matches:
    raise ValueError("No 'REPORT:' headings found. Please confirm your big file format contains lines like: REPORT: Report_10_uml_pages.pdf")

# Save each report block into its own .txt file
index_lines = []
written = 0

for m in matches:
    report_name = m.group("report").strip()
    body = m.group("body")

    # Use the report heading as filename (safe)
    # Example: "Report_10_uml_pages.pdf" -> "Report_10_uml_pages.txt"
    safe_stem = Path(report_name).stem  # removes ".pdf" safely
    out_path = OUT_SUB / f"{safe_stem}.txt"

    # Reconstruct block exactly (no truncation / no modification)
    block = f"REPORT: {report_name}\n{body}".strip() + "\n"

    out_path.write_text(block, encoding="utf-8")
    written += 1
    index_lines.append(f"{report_name} -> {out_path.name}")

# Save an index file so you can verify mapping quickly
index_path = OUT_ROOT / "index.txt"
index_path.write_text("\n".join(index_lines) + "\n", encoding="utf-8")

print(f"✅ Saved {written} separate report TXT files in: {OUT_SUB}")
print(f"🧾 Index saved at: {index_path}")
